In [ ]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q transformers datasets accelerate scikit-learn pillow

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report
)

from torch.utils.data import Dataset

from transformers import (

    BertTokenizerFast,

    Trainer,

    TrainingArguments
)

from transformers.models.visual_bert.modeling_visual_bert import (

    VisualBertModel
)

In [ ]:
# ============================================================
# FORCE CPU MODE
# ============================================================

torch.cuda.is_available = lambda : False

DEVICE = torch.device("cpu")

print(DEVICE)

In [ ]:
# ============================================================
# DOWNLOAD MAMI DATASET
# ============================================================

import kagglehub

path = kagglehub.dataset_download(

    "nrizwan/mami-dataset"
)

print(path)

In [ ]:
# ============================================================
# DEFINE PATHS
# ============================================================

DATASET_DIR = os.path.join(

    path,

    "MAMI_Dataset",

    "data"
)

TRAIN_FILE = os.path.join(

    DATASET_DIR,

    "train.tsv"
)

DEV_FILE = os.path.join(

    DATASET_DIR,

    "validation.tsv"
)

TEST_FILE = os.path.join(

    DATASET_DIR,

    "test.tsv"
)

TRAIN_IMAGE_DIR = os.path.join(

    DATASET_DIR,

    "MAMI_2022_images",

    "training_images"
)

TEST_IMAGE_DIR = os.path.join(

    DATASET_DIR,

    "MAMI_2022_images",

    "test_images"
)

print(TRAIN_FILE)

print(DEV_FILE)

print(TEST_FILE)

In [ ]:
# ============================================================
# LOAD TSV FILES
# ============================================================

train_df = pd.read_csv(

    TRAIN_FILE,

    sep="\t"
)

dev_df = pd.read_csv(

    DEV_FILE,

    sep="\t"
)

test_df = pd.read_csv(

    TEST_FILE,

    sep="\t"
)

print("TRAIN:", len(train_df))

print("DEV:", len(dev_df))

print("TEST:", len(test_df))

In [ ]:
# ============================================================
# CHECK COLUMNS
# ============================================================

print(train_df.columns)

In [ ]:
# ============================================================
# RENAME COLUMNS CORRECTLY
# ============================================================

train_df = train_df.rename(columns={

    "file_name": "image"
})

dev_df = dev_df.rename(columns={

    "file_name": "image"
})

test_df = test_df.rename(columns={

    "file_name": "image"
})

print(train_df.columns)

print(dev_df.columns)

print(test_df.columns)

In [ ]:
# ============================================================
# KEEP ONLY REQUIRED COLUMNS
# ============================================================

train_df = train_df[[

    "image",

    "text",

    "label"
]]

dev_df = dev_df[[

    "image",

    "text",

    "label"
]]

test_df = test_df[[

    "image",

    "text",

    "label"
]]

In [ ]:
# ============================================================
# LIMIT DATASET SIZE
# ============================================================

train_df = train_df.sample(

    n=min(1000, len(train_df)),

    random_state=42
)

dev_df = dev_df.sample(

    n=min(500, len(dev_df)),

    random_state=42
)

test_df = test_df.sample(

    n=min(500, len(test_df)),

    random_state=42
)

train_df = train_df.reset_index(drop=True)

dev_df = dev_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

print(len(train_df))

print(len(dev_df))

print(len(test_df))

In [ ]:
# ============================================================
# RESET INDICES
# ============================================================

train_df = train_df.reset_index(drop=True)

dev_df = dev_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

In [ ]:
# ============================================================
# LOAD TOKENIZER
# ============================================================

tokenizer = BertTokenizerFast.from_pretrained(

    "bert-base-uncased"
)

In [ ]:
# ============================================================
# CREATE DATASET CLASS
# ============================================================

class MAMIDataset(Dataset):

    def __init__(

        self,

        dataframe,

        tokenizer,

        image_dir,

        max_length=128
    ):

        self.df = dataframe

        self.tokenizer = tokenizer

        self.image_dir = image_dir

        self.max_length = max_length

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        sample = self.df.iloc[idx]

        text = str(sample["text"])

        label = int(sample["label"])

        image_path = os.path.join(

            self.image_dir,

            sample["image"]
        )

        image = Image.open(

            image_path

        ).convert("RGB")

        image = image.resize((224, 224))

        image = np.array(image).astype(np.float32)

        image = image / 255.0

        visual_embeds = torch.tensor(

            image

        ).permute(2, 0, 1)

        visual_embeds = visual_embeds.mean(dim=0)

        visual_embeds = visual_embeds.flatten()

        visual_embeds = visual_embeds[:2048]

        if visual_embeds.shape[0] < 2048:

            pad_size = (

                2048 -

                visual_embeds.shape[0]
            )

            visual_embeds = torch.cat([

                visual_embeds,

                torch.zeros(pad_size)
            ])

        visual_embeds = visual_embeds.unsqueeze(0)

        visual_attention_mask = torch.ones(

            visual_embeds.shape[:-1],

            dtype=torch.float
        )

        encoding = self.tokenizer(

            text,

            padding="max_length",

            truncation=True,

            max_length=self.max_length,

            return_tensors="pt"
        )

        return {

            "input_ids":

            encoding["input_ids"].squeeze(0),

            "attention_mask":

            encoding["attention_mask"].squeeze(0),

            "visual_embeds":

            visual_embeds,

            "visual_attention_mask":

            visual_attention_mask,

            "labels":

            torch.tensor(

                label,

                dtype=torch.long
            )
        }

In [ ]:
# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = MAMIDataset(

    train_df,

    tokenizer,

    TRAIN_IMAGE_DIR
)

dev_dataset = MAMIDataset(

    dev_df,

    tokenizer,

    TRAIN_IMAGE_DIR
)

test_dataset = MAMIDataset(

    test_df,

    tokenizer,

    TEST_IMAGE_DIR
)

In [ ]:
# ============================================================
# CREATE VISUALBERT CLASSIFIER
# ============================================================

class VisualBERTClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.visualbert = VisualBertModel.from_pretrained(

            "uclanlp/visualbert-vqa-coco-pre"
        )

        hidden_size = (

            self.visualbert.config.hidden_size
        )

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(

            hidden_size,

            2
        )

    def forward(

        self,

        input_ids,

        attention_mask,

        visual_embeds,

        visual_attention_mask,

        labels=None
    ):

        outputs = self.visualbert(

            input_ids=input_ids,

            attention_mask=attention_mask,

            visual_embeds=visual_embeds,

            visual_attention_mask=

            visual_attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        pooled_output = self.dropout(

            pooled_output
        )

        logits = self.classifier(

            pooled_output
        )

        loss = None

        if labels is not None:

            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(

                logits,

                labels
            )

        return {

            "loss": loss,

            "logits": logits
        }

In [ ]:
# ============================================================
# INITIALIZE MODEL
# ============================================================

model = VisualBERTClassifier()

In [ ]:
# ============================================================
# DEFINE METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(

        logits,

        axis=1
    )

    accuracy = accuracy_score(

        labels,

        preds
    )

    precision = precision_score(

        labels,

        preds
    )

    recall = recall_score(

        labels,

        preds
    )

    f1 = f1_score(

        labels,

        preds
    )

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1
    }

In [ ]:
# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./visualbert_mami",

    num_train_epochs=1,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_steps=10,

    report_to="none"
)

In [ ]:
# ============================================================
# CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=dev_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
# ============================================================
# TRAIN MODEL
# ============================================================

start_train_time = time.time()

trainer.train()

end_train_time = time.time()

training_time = (

    end_train_time -

    start_train_time
)

print(

    f"Training Time: "

    f"{training_time:.2f} seconds"
)

In [ ]:
# ============================================================
# EVALUATE MODEL
# ============================================================

predictions = trainer.predict(

    test_dataset
)

logits = predictions.predictions

labels = predictions.label_ids

preds = np.argmax(

    logits,

    axis=1
)

In [ ]:
# ============================================================
# PERFORMANCE METRICS
# ============================================================

accuracy = accuracy_score(

    labels,

    preds
)

precision = precision_score(

    labels,

    preds
)

recall = recall_score(

    labels,

    preds
)

f1 = f1_score(

    labels,

    preds
)

cm = confusion_matrix(

    labels,

    preds
)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1 Score:", f1)

print()

print(cm)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(

    classification_report(

        labels,

        preds,

        digits=4
    )
)

In [ ]:
# ============================================================
# INFERENCE TIME
# ============================================================

start_inference = time.time()

_ = trainer.predict(test_dataset)

end_inference = time.time()

total_inference_time = (

    end_inference -

    start_inference
)

average_inference_time = (

    total_inference_time /

    len(test_dataset)
)

print(

    f"Total Inference Time: "

    f"{total_inference_time:.4f} seconds"
)

print(

    f"Average Time Per Sample: "

    f"{average_inference_time:.6f} seconds"
)